# Imbalance Handling

Fraud detection datasets are highly imbalanced.

This notebook prepares imbalance handling methods for model training.

Main methods:

1. Class weights
2. XGBoost `scale_pos_weight`
3. CatBoost class weights
4. Random undersampling

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.imbalance import (
    load_processed_train_test_data,
    calculate_target_distribution,
    calculate_class_weights,
    calculate_xgboost_scale_pos_weight,
    create_random_undersampled_training_set,
    run_imbalance_preparation,
)

pd.set_option("display.max_columns", None)

In [2]:
X_train, X_test, y_train, y_test = load_processed_train_test_data()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5090096, 28)
X_test: (1272524, 28)
y_train: (5090096,)
y_test: (1272524,)


In [4]:
train_distribution = calculate_target_distribution(y_train)

train_distribution

{'class_counts': {'0': 5083526, '1': 6570},
 'class_percentages': {'0': 99.87092581357993, '1': 0.12907418642005966}}

In [5]:
test_distribution = calculate_target_distribution(y_test)

test_distribution

{'class_counts': {'0': 1270881, '1': 1643},
 'class_percentages': {'0': 99.87088652159017, '1': 0.12911347840983745}}

In [6]:
distribution_table = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_percentage": y_train.value_counts(normalize=True).sort_index() * 100,
    "test_count": y_test.value_counts().sort_index(),
    "test_percentage": y_test.value_counts(normalize=True).sort_index() * 100,
})

distribution_table

,train_count,train_percentage,test_count,test_percentage
is_fraud,,,,
0,5083526,99.870926,1270881,99.870887
1,6570,0.129074,1643,0.129113


In [7]:
class_weights = calculate_class_weights(y_train)

class_weights

{0: 0.5006462050159672, 1: 387.37412480974126}

In [8]:
scale_pos_weight = calculate_xgboost_scale_pos_weight(y_train)

scale_pos_weight

773.7482496194825

In [9]:
X_train_under, y_train_under = create_random_undersampled_training_set(
    X_train,
    y_train,
    sampling_strategy=0.2,
)

print("Original X_train shape:", X_train.shape)
print("Undersampled X_train shape:", X_train_under.shape)

Original X_train shape: (5090096, 28)
Undersampled X_train shape: (39420, 28)


In [10]:
undersampled_distribution = pd.DataFrame({
    "count": y_train_under.value_counts().sort_index(),
    "percentage": y_train_under.value_counts(normalize=True).sort_index() * 100,
})

undersampled_distribution

,count,percentage
is_fraud,,
0,32850,83.333333
1,6570,16.666667


In [11]:
imbalance_report = run_imbalance_preparation(create_undersampled_data=True)

imbalance_report

Loading processed train/test data...

Train target distribution:
{'class_counts': {'0': 5083526, '1': 6570}, 'class_percentages': {'0': 99.87092581357993, '1': 0.12907418642005966}}

Test target distribution:
{'class_counts': {'0': 1270881, '1': 1643}, 'class_percentages': {'0': 99.87088652159017, '1': 0.12911347840983745}}

Calculating class weights...
Class weights: {0: 0.5006462050159672, 1: 387.37412480974126}

Calculating XGBoost scale_pos_weight...
scale_pos_weight: 773.7482496194825

Creating random undersampled training set...
Original training shape: (5090096, 28)
Undersampled training shape: (39420, 28)

Undersampled target distribution:
{'class_counts': {'0': 32850, '1': 6570}, 'class_percentages': {'0': 83.33333333333334, '1': 16.666666666666664}}
Undersampled training data saved:
C:\Users\User\Documents\fraud-detection-system\data\processed\X_train_under.csv
C:\Users\User\Documents\fraud-detection-system\data\processed\y_train_under.csv
Imbalance report saved to: C:\Users\

{'train_distribution': {'class_counts': {'0': 5083526, '1': 6570},
  'class_percentages': {'0': 99.87092581357993, '1': 0.12907418642005966}},
 'test_distribution': {'class_counts': {'0': 1270881, '1': 1643},
  'class_percentages': {'0': 99.87088652159017, '1': 0.12911347840983745}},
 'class_weights_for_sklearn': {0: 0.5006462050159672, 1: 387.37412480974126},
 'xgboost_scale_pos_weight': 773.7482496194825,
 'catboost_class_weights': [0.5006462050159672, 387.37412480974126],
 'notes': ['The dataset is highly imbalanced.',
  'Accuracy should not be used as the main metric.',
  'Use recall, precision, F1-score, ROC-AUC, PR-AUC, and confusion matrix.',
  'Resampling must be applied only to training data, never test data.'],
 'undersampled_train_distribution': {'class_counts': {'0': 32850, '1': 6570},
  'class_percentages': {'0': 83.33333333333334, '1': 16.666666666666664}},
 'undersampling_strategy': 'RandomUnderSampler with sampling_strategy=0.2'}

## Imbalance Handling Notes

The dataset is highly imbalanced, meaning normal transactions are much more common than fraud transactions.

Because of this, accuracy is not a reliable metric.

For model training, the following strategies will be tested:

### Class Weights

Class weights give more importance to the minority fraud class during training.

Used for:

- Logistic Regression
- Random Forest
- CatBoost

### XGBoost scale_pos_weight

`scale_pos_weight` tells XGBoost how much more weight to give to the positive fraud class.

Formula:

```text
negative samples / positive samples